In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.3f')

In [ ]:
# show max and min values a datatype can hold
import numpy as np
print(np.iinfo('int8'))
print(np.iinfo('int16'))
print(np.iinfo('int32'))
print(np.iinfo('int64'))

Machine parameters for int8
---------------------------------------------------------------
min = -128
max = 127
---------------------------------------------------------------

Machine parameters for int16
---------------------------------------------------------------
min = -32768
max = 32767
---------------------------------------------------------------

Machine parameters for int32
---------------------------------------------------------------
min = -2147483648
max = 2147483647
---------------------------------------------------------------

Machine parameters for int64
---------------------------------------------------------------
min = -9223372036854775808
max = 9223372036854775807
---------------------------------------------------------------



In [ ]:
def reduce_mem_usage(df, verbose=True):
  numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
  start_mem = df.memory_usage().sum() / 1024**2
  for col in df.columns:
    col_type = df[col].dtypes
    if col_type in numerics:
      c_min = df[col].min()
      c_max = df[col].max()

      if str(col_type)[:3] == 'int':
        if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
          df[col] = df[col].astype(np.int8)

        elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
          df[col] = df[col].astype(np.int16)

        elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
          df[col] = df[col].astype(np.int32)

        elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
          df[col] = df[col].astype(np.int64)

      else:
        if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
          df[col] = df[col].astype(np.float16)

        elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
          df[col] = df[col].astype(np.float32)

        else:
          df[col] = df[col].astype(np.float64)

  end_mem = df.memory_usage(deep = True).sum() / 1024**2
  if verbose:
    print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(
        end_mem, 100 * (start_mem - end_mem) / start_mem))
  return df

In [ ]:
df = pd.DataFrame({'age': [1,2,3,4,5,6]})
print(df.dtypes)
df.memory_usage() # 64 bits per item

age    int64
dtype: object


,0
Index,132
age,48


In [ ]:
df_compressed = reduce_mem_usage(df)
print(df_compressed.dtypes)
df.memory_usage() # bits per item

Mem. usage decreased to  0.00 Mb (23.3% reduction)
age    int8
dtype: object


,0
Index,132
age,6


In [ ]:
# Create sparse matrix using csr_matrix()
import numpy as np
from scipy.sparse import csr_matrix

row = np.array([0, 0, 1, 2, 1])
col = np.array([0, 1, 0, 2, 2])

# data (the non-zero values)
data = np.array([1, 2, 3, 4, 5])

# creating sparse matrix
sparseMatrix = csr_matrix((data, (row, col)), shape=(3, 3))

# print the sparse matrix
print(sparseMatrix)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5 stored elements and shape (3, 3)>
  Coords	Values
  (0, 0)	1
  (0, 1)	2
  (1, 0)	3
  (1, 2)	5
  (2, 2)	4


In [ ]:
s_array = sparseMatrix.toarray()
s_array

array([[1, 2, 0],
       [3, 0, 5],
       [0, 0, 4]])

In [ ]:
sm = csr_matrix(s_array)
sm

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5 stored elements and shape (3, 3)>

In [ ]:
print(sm)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5 stored elements and shape (3, 3)>
  Coords	Values
  (0, 0)	1
  (0, 1)	2
  (1, 0)	3
  (1, 2)	5
  (2, 2)	4


In [ ]:
df_random = pd.DataFrame(np.random.randint(0, 100, size=(10000000, 10)))
df_random.to_csv("data_random.csv")

In [ ]:
%%time
df = pd.read_csv('data_random.csv')

CPU times: user 5.05 s, sys: 985 ms, total: 6.03 s
Wall time: 6.04 s


In [ ]:
!pip install pyarrow

In [ ]:
%%time
df.to_csv("data_random.csv")

CPU times: user 40.8 s, sys: 760 ms, total: 41.6 s
Wall time: 47 s


In [ ]:
%%time
df.to_parquet("data_random.parquet")

CPU times: user 2.4 s, sys: 203 ms, total: 2.6 s
Wall time: 2.69 s


In [ ]:
%%time
df.to_feather("data_random.feather")

CPU times: user 1.69 s, sys: 335 ms, total: 2.03 s
Wall time: 2.51 s


In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix

df = pd.read_csv('/content/Week 1_movie_ratings_grid_Dataset.csv', index_col = 0)
print('Shape:', df.shape)
print(df.iloc[:6, :8])

Shape: (100, 30)
          The_Dark_Knight  Inception  Interstellar  The_Matrix  Avengers_Endgame  Parasite  Joker  1917
User_001                0          0             0           0                 0         0      0     0
User_002                0          0             0           0                 0         0      0     0
User_003                0          5             4           3                 4         5      0     0
User_004                0          5             0           2                 0         3      0     0
User_005                0          0             0           0                 0         0      0     0
User_006                0          0             0           0                 0         0      0     0


In [ ]:
mem = df.memory_usage(deep = True)
print(mem)
print(f"Total: {mem.sum():,} bytes ({mem.sum()/1024:.2f}KB)")

Index                    5700
The_Dark_Knight           800
Inception                 800
Interstellar              800
The_Matrix                800
Avengers_Endgame          800
Parasite                  800
Joker                     800
1917                      800
Ford_v_Ferrari            800
Once_Upon_Hollywood       800
SpiderMan_NWH             800
Dune                      800
The_Batman                800
Top_Gun_Maverick          800
Everything_Everywhere     800
Avatar_WoW                800
Black_Panther             800
Doctor_Strange            800
Thor_LandT                800
Oppenheimer               800
Barbie                    800
Killers_Flower_Moon       800
Past_Lives                800
Saltburn                  800
Poor_Things               800
Zone_of_Interest          800
Maestro                   800
Priscilla                 800
Napoleon                  800
May_December              800
dtype: int64
Total: 29,700 bytes (29.00KB)


In [ ]:
total = df.shape[0] * df.shape[1]
zeros = (df == 0).sum().sum()
nonzeros = total - zeros

print(f"Total cells    : {total:,}")
print(f"Zeros cells    : {zeros:,} ({100 * zeros / total:.1f}%) <-  wasted!")
print(f"Non-zeros cells: {nonzeros:,} ({100 * nonzeros / total:.1f}%) <-  real data")

Total cells    : 3,000
Zeros cells    : 2,477 (82.6%) <-  wasted!
Non-zeros cells: 523 (17.4%) <-  real data


In [ ]:
sparse = csr_matrix(df.values)

sparse_bytes = (sparse.data.nbytes
                + sparse.indices.nbytes
                + sparse.indptr.nbytes)

print(sparse)



<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 523 stored elements and shape (100, 30)>
  Coords	Values
  (0, 8)	5
  (0, 9)	3
  (0, 15)	3
  (0, 17)	4
  (0, 23)	4
  (0, 24)	4
  (0, 27)	2
  (0, 29)	4
  (1, 15)	5
  (1, 19)	5
  (1, 26)	5
  (1, 29)	5
  (2, 1)	5
  (2, 2)	4
  (2, 3)	3
  (2, 4)	4
  (2, 5)	5
  (2, 9)	2
  (2, 10)	2
  (3, 1)	5
  (3, 3)	2
  (3, 5)	3
  (3, 15)	3
  (3, 17)	3
  (3, 18)	4
  :	:
  (95, 28)	5
  (96, 13)	5
  (96, 17)	3
  (96, 18)	4
  (96, 19)	4
  (97, 1)	4
  (97, 6)	3
  (97, 7)	2
  (97, 13)	3
  (97, 16)	5
  (97, 22)	3
  (97, 29)	5
  (98, 4)	5
  (98, 8)	3
  (98, 9)	4
  (98, 16)	1
  (98, 17)	5
  (98, 19)	5
  (98, 25)	4
  (99, 7)	5
  (99, 12)	5
  (99, 15)	4
  (99, 18)	5
  (99, 22)	4
  (99, 26)	4


In [ ]:
print(f"DataFrame memory : {mem.sum():,} bytes ({mem.sum()/1024:.2f}KB)")
print(f"Sparse matrix    : {sparse_bytes:,} bytes ({sparse_bytes/1024:.2f}KB)")
print(f"Memory saved     : {(mem.sum() - sparse_bytes) / 1024:.2f}KB")
print(f"Sparse is        : {mem.sum()/sparse_bytes: 1f}x smaller")

DataFrame memory : 29,700 bytes (29.00KB)
Sparse matrix    : 6,680 bytes (6.52KB)
Memory saved     : 22.48KB
Sparse is        :  4.446108x smaller
